### 79.单词搜索
给定一个 m x n 二维字符网格 board 和一个字符串单词 word 。如果 word 存在于网格中，返回 true ；否则，返回 false 。

单词必须按照字母顺序，通过相邻的单元格内的字母构成，其中“相邻”单元格是那些水平相邻或垂直相邻的单元格。同一个单元格内的字母不允许被重复使用。

示例 1：

输入：board = [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]], word = "ABCCED"

输出：true

示例 2：

输入：board = [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]], word = "SEE"

输出：true


#### 1.回溯 + 图论
**思路**
1. 枚举 i=0,1,2,…,m−1 和 j=0,1,2,…,n−1，以 (i,j) 为起点开始搜索。
2. 同时，我们还需要知道当前匹配到了 word 的第几个字母(下标)，所以还需要一个参数 k。
3. 定义 dfs(i,j,k) 表示当前在 board[i][j] 这个格子，要匹配 word[k]，返回在这个状态下最终能否匹配成功（搜索成功）。
   1. if board[i][j] != word[k]: return False ,匹配失败
   2. 否则，if k == len(word) - 1: return True ,匹配成功
   3. else: 枚举 (i,j) 四周相邻格子(x,y), 如果 (x,y) 位置没有出界，则递归dfs(x,y,k+1)，如果搜索成功，则返回True，且 dfs(x,y,k) 也返回True
   4. 如果 递归四个格子都没有搜索成功，则返回False。

**注意**:
1. 搜索过程中，不能重复访问同一个格子。
2. 为了避免重复访问同一个格子，我们可以用一个二维数组 `vis[i][j]` 表示格子 (i,j) 是否被访问过。
3. 更简单的做法是，直接修改 `board[i][j]`，将其置为空（或者 0），返回 false 前再恢复成原来的值（恢复现场）。注意返回 true 的时候就不用恢复现场了，因为已经成功搜到 word 了。

优化：
1. 剪枝：统计字符数，如果 word 的某个字母的出现次数，比 board 中的这个字母的出现次数还要多，可以直接返回 false。
2. 从前找，或从后找：
   1. 如果 word=abcd 但 board 中的 a 很多，d 很少（比如只有一个），那么从 d 开始搜索，能更快地找到答案。
   2. 设 word 的第一个字母在 board 中出现了 x 次，word 的最后一个字母在 board 中出现了 y 次。如果 `y < x`，我们可以把 word 反转。



In [3]:
from typing import List
from collections import Counter
class Solution:
    def exist(self, board: List[List[str]], word: str) -> bool:
        
        cnt = Counter(c for row in board for c in row)
        # 优化1
        if not cnt >= Counter(word): # 如果存在 word中字符数大于 board
            return False

        # 优化2
        if cnt[word[-1]] < cnt[word[0]]:
            word = word[::-1]
        
        m, n = len(board), len(board[0])

        # i，j表示坐标位置， k 记录当前判断的word下标
        def dfs(i: int, j: int, k: int) -> bool:
            if board[i][j] != word[k]: # 匹配失败 
                return False
            
            if k == len(word) - 1: 
                return True
            
            # word[k]匹配成功，标记[i,j]访问过
            board[i][j] = '' # 清空，标记访问过
            
            for x, y in (i, j - 1), (i, j + 1), (i - 1, j), (i + 1, j):  # 相邻格子
                # 搜索成功
                if 0 <= x <m and 0 <= y < n and dfs(x, y, k+1):
                    return True
            
            board[i][j] = word[k] #恢复现场
            return False # 没搜到

        # 只要从一个起始位置 搜索成功，则成功，否则失败
        return any(dfs(i,j,0) for i in range(m) for j in range(n))
    
board = [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]]
# word = "ABCCED"
# word = "SEE"
word = "ABCB"
print(Solution().exist(board, word))


False
